# Phase F: Retrieval Specialist LoRA Domain Fine-Tuning
### AI Search Framework — Parameter Adaptation Initiative

**Model:** `microsoft/Phi-3.5-mini-instruct` (3.82B parameters)
**Target Specialist:** `retrieval_qa` (Technical RFC Standards & Linux Kernel Security)
**Target Platform:** Google Colab (Free Tier Tesla T4 15.3 GB VRAM)
**Hardware Cost:** $0.00

This notebook implements:
1. **Incremental Checkpoint Safety:** Saves checkpoints every 25 steps directly to mounted Google Drive. Auto-resumes from latest checkpoint upon disconnect.
2. **LoRA Adapter Fine-Tuning:** 4-bit QLoRA with rank=16, alpha=32 on all linear projections.
3. **Weight Fusion & GGUF Export:** Merges adapter into base weights and converts to `Q4_K_M` GGUF format for local Ollama evaluation.

In [ ]:
# Step 1: Mount Google Drive for Persistent Checkpoint Safety
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ai_search_phase_f'
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
EXPORT_DIR = os.path.join(DRIVE_DIR, 'exported_gguf')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Checkpoint directory configured on persistent Google Drive: {CHECKPOINT_DIR}")

In [ ]:
# Step 2: Install Fine-Tuning & Export Dependencies
!pip install -q torch transformers datasets peft bitsandbytes accelerate trl
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
print("Dependencies installed successfully.")

In [ ]:
# Step 3: Load Base Model in 4-bit NF4 Quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model = prepare_model_for_kbit_training(base_model)

# Target all linear attention and MLP projection layers
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()
print("Base model and LoRA adapter initialized.")

In [ ]:
# Step 4: Load and Format Training Dataset
# (Upload 'retrieval_qa_finetune_dataset.json' to Colab or load from Drive)
import json
from datasets import Dataset

dataset_path = os.path.join(DRIVE_DIR, 'retrieval_qa_finetune_dataset.json')
if not os.path.exists(dataset_path):
    dataset_path = 'retrieval_qa_finetune_dataset.json'

with open(dataset_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

formatted_samples = []
for item in raw_data:
    formatted_prompt = f"<|system|>\n{item['system']}<|end|>\n<|user|>\n{item['prompt']}<|end|>\n<|assistant|>\n{item['response']}<|end|>"
    formatted_samples.append({"text": formatted_prompt})

dataset = Dataset.from_list(formatted_samples)
print(f"Loaded {len(dataset)} formatted training samples.")

In [ ]:
# Step 5: Checkpoint Safety Training Configuration with Auto-Resume
from trl import SFTTrainer
from transformers import TrainingArguments

# Check for existing checkpoints on Google Drive to resume from
checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')]
resume_checkpoint = None
if checkpoints:
    latest_step = max([int(c.split('-')[1]) for c in checkpoints])
    resume_checkpoint = os.path.join(CHECKPOINT_DIR, f'checkpoint-{latest_step}')
    print(f"DISCONNECT RECOVERY: Found existing checkpoint on Google Drive: {resume_checkpoint}")
    print("Training will automatically resume from this step.")
else:
    print("No prior checkpoint found. Starting fresh training run.")

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    num_train_epochs=3,
    max_steps=-1,
    save_strategy="steps",
    save_steps=25,               # Save every 25 steps to Google Drive
    save_total_limit=3,          # Keep 3 latest checkpoints
    fp16=True,
    optim="paged_adamw_8bit",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1536,
    tokenizer=tokenizer,
    args=training_args
)

# Execute training with explicit resume capability
trainer.train(resume_from_checkpoint=resume_checkpoint)

# Save final adapter weights to Google Drive
FINAL_ADAPTER_DIR = os.path.join(DRIVE_DIR, 'final_adapter')
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Final adapter weights saved to Google Drive: {FINAL_ADAPTER_DIR}")

In [ ]:
# Step 6: Full Weight Fusion & Export to 16-bit Model
# Merging adapter into base weights creates a standalone model compatible with llama.cpp
from peft import PeftModel

print("Loading base model in FP16 for clean mathematical weight fusion...")
del model, base_model, trainer
torch.cuda.empty_cache()

base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(base_model_fp16, FINAL_ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

MERGED_DIR = "/content/phi35_merged_fp16"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged FP16 model saved to {MERGED_DIR}")

In [ ]:
# Step 7: Convert Merged Model to Quantized GGUF for Local Ollama
GGUF_FP16_PATH = "/content/phi35_ft.fp16.gguf"
FINAL_GGUF_PATH = os.path.join(EXPORT_DIR, "phi3.5_ft_retrieval_q4_k_m.gguf")

print("Converting merged HuggingFace weights to GGUF format...")
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_FP16_PATH} --outtype f16

print("Quantizing to Q4_K_M for efficient Ollama CPU/GPU execution...")
!cd /content/llama.cpp && cmake -B build && cmake --build build --config Release -t llama-quantize
!/content/llama.cpp/build/bin/llama-quantize {GGUF_FP16_PATH} {FINAL_GGUF_PATH} q4_k_m

print(f"SUCCESS: Standalone quantized GGUF exported to Google Drive: {FINAL_GGUF_PATH}")
print(f"File size: {os.path.getsize(FINAL_GGUF_PATH) / 1e9:.2f} GB")

### Step 8: Local Deployment into Ollama Pipeline
Once `phi3.5_ft_retrieval_q4_k_m.gguf` is downloaded to your local repository:
1. Move the file into `models/phi3.5_ft_retrieval_q4_k_m.gguf`.
2. Create a local `Modelfile`:
   ```dockerfile
   FROM ./models/phi3.5_ft_retrieval_q4_k_m.gguf
   TEMPLATE "{{ if .System }}<|system|>\n{{ .System }}<|end|>\n{{ end }}{{ if .Prompt }}<|user|>\n{{ .Prompt }}<|end|>\n{{ end }}<|assistant|>\n{{ .Response }}<|end|>"
   PARAMETER stop "<|system|>"
   PARAMETER stop "<|user|>"
   PARAMETER stop "<|end|>"
   PARAMETER stop "<|assistant|>"
   PARAMETER temperature 0.0
   ```
3. Build the Ollama model: `ollama create phi3.5-ft-retrieval -f Modelfile`.
4. Test in local pipeline via `OllamaModelRunner(api_model_name="phi3.5-ft-retrieval:latest")`.